# END-TO-END PDF TO RAG PIPELINE
Complete workflow from PDF to JSON chunks with metadata

In [ ]:
# SETUP: Load environment and imports
import sys
from pathlib import Path

# Add project root to path so we can import from src/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

import os
from src.ingestion.pdf_extractor import organize_extracted_pdf, generate_image_descriptions, insert_image_descriptions, add_image_tags_to_document
from src.ingestion.extract_metadata import extract_metadata_with_llm
from src.ingestion.chunk_document import chunk_document_nltk, combine_chunks, save_combined_chunks, print_chunks_summary
from src.ingestion.attach_metadata_to_chunks import attach_metadata_to_chunks

print("✓ All imports loaded")
print(f"📁 Project root: {PROJECT_ROOT}")

In [ ]:
# CONFIGURE: Set your PDF path here
pdf_path = str(PROJECT_ROOT / "data" / "raw" / "silver_report.pdf")  # CHANGE THIS
output_base_folder = str(PROJECT_ROOT / "data" / "processed")

print(f"📄 Input PDF: {pdf_path}")
print(f"📁 Output folder: {output_base_folder}")
print(f"✓ Configuration ready\n")

In [ ]:
# STEP 1: Extract text, images, and metadata
print(f"{'='*60}")
print("STEP 1: Extract text, images, and metadata")
print(f"{'='*60}\n")

folder = organize_extracted_pdf(pdf_path, output_base_folder)

if folder:
    print(f"✓ Extraction complete: {folder}\n")
else:
    print("❌ Extraction failed")

In [ ]:
# STEP 2: Extract metadata using LLM
print(f"{'='*60}")
print("STEP 2: Extract metadata (title, date, theme)")
print(f"{'='*60}\n")

metadata_txt = os.path.join(folder, "metadata.txt")
metadata_json = os.path.join(folder, "metadata.json")

metadata = extract_metadata_with_llm(
    metadata_txt_path=metadata_txt,
    output_json_path=metadata_json,
    provider="groq"  # Free API with no quota limits
)

if metadata:
    print()
else:
    print("❌ Metadata extraction failed")

In [ ]:
# STEP 3: Generate image descriptions using Groq Vision
print(f"{'='*60}")
print("STEP 3: Generate image descriptions (Groq)")
print(f"{'='*60}\n")

generate_image_descriptions(
    folder,
    provider="groq",
    model_name="meta-llama/llama-4-scout-17b-16e-instruct"
)
print()

In [ ]:
# STEP 4: Add image location tags (convert [IMAGE_X] to [IMAGE_X_DESC_START/END])
print(f"{'='*60}")
print("STEP 4: Add image location tags to document")
print(f"{'='*60}\n")

document_path = os.path.join(folder, "document.txt")
add_image_tags_to_document(document_path)
print(f"✓ Image tags added\n")

In [ ]:
# STEP 5: Insert image descriptions into document
print(f"{'='*60}")
print("STEP 5: Insert image descriptions into document")
print(f"{'='*60}\n")

insert_image_descriptions(folder)
print()

In [ ]:
# STEP 6: Chunk document using NLTK
print(f"{'='*60}")
print("STEP 6: Chunk document using NLTK sentence tokenizer")
print(f"{'='*60}\n")

chunks_folder = os.path.join(folder, "chunks")
chunks = chunk_document_nltk(document_path, chunks_folder)

if chunks:
    print()
else:
    print("❌ Chunking failed")

In [ ]:
# STEP 7: Combine chunks to 300-word max
print(f"{'='*60}")
print("STEP 7: Combine chunks to 300-word max")
print(f"{'='*60}\n")

combined_chunks = combine_chunks(chunks, max_words=300)
print(f"Combined {len(chunks)} chunks → {len(combined_chunks)} chunks\n")

In [ ]:
# STEP 8: Save combined chunks
print(f"{'='*60}")
print("STEP 8: Save combined chunks")
print(f"{'='*60}\n")

chunks_final_folder = os.path.join(folder, "chunks_final")
save_combined_chunks(combined_chunks, chunks_final_folder)
print()

In [ ]:
# STEP 9: Attach metadata to chunks
print(f"{'='*60}")
print("STEP 9: Attach metadata to all chunks")
print(f"{'='*60}\n")

final_chunks = attach_metadata_to_chunks(
    chunks_folder=chunks_final_folder,
    metadata_json_path=metadata_json,
    output_folder=folder
)

if final_chunks:
    print()
else:
    print("❌ Failed to attach metadata")

In [ ]:
# SUMMARY: Display results
print(f"{'='*60}")
print("✅ PIPELINE COMPLETE")
print(f"{'='*60}\n")

print_chunks_summary(final_chunks)

output_json = os.path.join(folder, "chunks_with_metadata.json")
print(f"\n📁 Final output files:")
print(f"   JSON: {output_json}")
print(f"   Chunks: {os.path.join(folder, 'chunks_with_metadata')}/")
print(f"\n✅ Ready for vector embedding!")

In [ ]:
# OPTIONAL: Inspect results
import json

# Read the final JSON
with open(output_json, "r") as f:
    chunks_data = json.load(f)

print(f"Total chunks: {len(chunks_data)}\n")

# Show first chunk as example
if chunks_data:
    first_chunk = chunks_data[0]
    print("=" * 60)
    print(f"EXAMPLE: Chunk {first_chunk['chunk_id']}")
    print("=" * 60)
    print(f"\nMetadata:")
    print(f"  Title: {first_chunk['metadata']['title'][:60]}...")
    print(f"  Date: {first_chunk['metadata']['date']}")
    print(f"  Theme: {first_chunk['metadata']['theme']}")
    print(f"\nContent ({first_chunk['word_count']} words):")
    print(f"  {first_chunk['content'][:200]}...")